<a href="https://colab.research.google.com/github/Harshal5167/ExerciseApp/blob/main/nb/Mistral_v0.3_(7B)-CPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
from unsloth import is_bfloat16_supported
import torch
print(f"BFloat16 supported: {is_bfloat16_supported()}")

# Explicitly set the data type when loading the model
# If bfloat16 is supported, use it; otherwise use float16
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"Using dtype: {dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
BFloat16 supported: False
Using dtype: torch.float16


### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

We also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.

In [4]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.4.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [6]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

<a name="Data"></a>
### Data Prep
We now use the Korean subset of the [Wikipedia dataset](https://huggingface.co/datasets/wikimedia/wikipedia) to first continually pretrain the model. You can use **any language** you like! Go to [Wikipedia's List of Languages](https://en.wikipedia.org/wiki/List_of_Wikipedias) to find your own language!

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_v0.3_(7B)-Conversational.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

**[NOTE]** Use https://translate.google.com to translate from English to Korean!

In [7]:
# Load custom dataset from raw text file in content directory
import os
from datasets import Dataset

# Function to read the raw text file
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Split the text into chunks (e.g., by paragraphs or with a fixed length)
    # Here we split by double newlines (paragraphs)
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks

# Path to your text file in the content directory
file_path = "/content/Review of Endodontics and Operative Dentistry by Garg, Nisha.txt"  # Change this to your actual file path

# Create a dataset from the text chunks
text_chunks = read_text_file(file_path)
dataset = Dataset.from_dict({"text": text_chunks})

# Apply formatting function to prepare data for training
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    outputs = []
    for text in examples["text"]:
        # Add EOS token to each chunk
        formatted_text = text + EOS_TOKEN
        outputs.append(formatted_text)
    return {"text": outputs}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2853 [00:00<?, ? examples/s]

We only use 1% of the dataset to speed things up! Use more for longer runs!

<a name="Train"></a>
### Continued Pretraining
Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 20 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate` to make continual pretraining work!

In [8]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    args = UnslothTrainingArguments(
        # Reduce batch size
        per_device_train_batch_size = 1,  # Reduced from 2 to 1
        gradient_accumulation_steps = 16, # Increased from 8 to 16 to maintain effective batch size

        max_steps = 10,
        warmup_steps = 10,

        # Keep learning rates the same
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5,

        fp16 = dtype == torch.float16,
        bf16 = dtype == torch.bfloat16,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # Add gradient checkpointing to save memory
        gradient_checkpointing = True,

        # Add memory optimization flags
        torch_compile = False,  # Disable torch.compile to reduce memory usage
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2853 [00:00<?, ? examples/s]

In [9]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.0 GB of memory reserved.


In [10]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,853 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 603,979,776/7,000,000,000 (8.63% trained)


Step,Training Loss
1,2.989900
2,3.058500
3,2.247400
4,2.388900
5,2.355900
6,2.185700
7,2.547500
8,2.426500
9,2.258800
10,2.297800


Unsloth: Will smartly offload gradients to save VRAM!


### Instruction Finetuning

We now use the [Alpaca in GPT4 Dataset](https://huggingface.co/datasets/FreedomIntelligence/alpaca-gpt4-korean) but translated in Korean!

Go to [vicgalle/alpaca-gpt4](https://huggingface.co/datasets/vicgalle/alpaca-gpt4) for the original GPT4 dataset for Alpaca or [MultilingualSIFT project](https://github.com/FreedomIntelligence/MultilingualSIFT) for other translations of the Alpaca dataset.

In [11]:
from datasets import load_dataset

alpaca_dataset = load_dataset("FreedomIntelligence/alpaca-gpt4-korean", split="train")

README.md:   0%|          | 0.00/124 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


alpaca-gpt4-korean.json:   0%|          | 0.00/51.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

We print 1 example:

In [12]:
print(alpaca_dataset[0])

{'conversations': [{'from': 'human', 'value': '재활용 캠페인 슬로건을 제시하세요.\n'}, {'from': 'gpt', 'value': '1. "더욱 녹색 미래를 위해 함께 줄이고, 재사용하고, 재활용하세요."\n2. "더 나은 내일을 위해 오늘 바로 재활용하세요."\n3. "쓰레기를 보물로 만드는 법 - 재활용!"\n4. "인생의 순환을 위해 재활용하세요."\n5. "자원을 아끼고 더 많이 재활용하세요."'}], 'id': '23712'}


We again use https://translate.google.com/ to translate the Alpaca format into Korean

In [14]:
import pandas as pd
from datasets import Dataset

# Load the Excel file
excel_path = "/content/Questions.xlsx"  # Change this to your actual file path
df = pd.read_excel(excel_path)

# Convert the dataframe to a Hugging Face dataset
instruction_dataset = Dataset.from_pandas(df)

# Create a prompt template for multiple-choice questions
mcq_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{question}

Options:
A. {option_a}
B. {option_b}
C. {option_c}
D. {option_d}

Select the correct answer and explain why.

### Response:
"""

# Format the instruction dataset for fine-tuning
def format_mcq_data(examples):
    texts = []

    for i in range(len(examples["Question"])):
        question = examples["Question"][i]
        option_a = examples["Option A"][i]
        option_b = examples["Option B"][i]
        option_c = examples["Option C"][i]
        option_d = examples["Option D"][i]
        correct_option = examples["Correct Option"][i]
        explanation = examples["Explanation"][i]

        # Format instruction
        instruction = mcq_prompt.format(
            question=question,
            option_a=option_a,
            option_b=option_b,
            option_c=option_c,
            option_d=option_d
        )

        # Format response
        response = f"The correct answer is {correct_option}.\n\n{explanation}"

        # Create final text with EOS token
        complete_text = instruction + response + tokenizer.eos_token
        texts.append(complete_text)

    return {"text": texts}

instruction_dataset = instruction_dataset.map(format_mcq_data, batched=True)


Map:   0%|          | 0/96 [00:00<?, ? examples/s]

We again employ `UnslothTrainer` and do instruction finetuning!

In [15]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = instruction_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        # Use num_train_epochs and warmup_ratio for longer runs!
        max_steps = 120,
        warmup_steps = 10,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.00,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/96 [00:00<?, ? examples/s]

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 96 | Num Epochs = 20 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 603,979,776/7,000,000,000 (8.63% trained)


Step,Training Loss
1,2.378100
2,2.327600
3,2.045800
4,1.601100
5,1.507400
6,1.152000
7,1.050300
8,0.938700
9,0.931700
10,0.914400


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

Remember to use https://translate.google.com/!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        # "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,", # instruction
        "피보나치 수열을 계속하세요: 1, 1, 2, 3, 5, 8,", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [19]:
# Test trained model on all questions with memory optimization

from unsloth import FastLanguageModel
import torch
import pandas as pd
import re
from tqdm import tqdm
import time
import gc

# Clear CUDA cache first to maximize available memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Current GPU memory usage: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

print("Loading model from saved checkpoint...")
# Load your trained model from the specific checkpoint directory
model_path = "/content/outputs/checkpoint-120"  # Specific checkpoint path

# Add proper device mapping to handle memory constraints
try:
    # First attempt: load with auto device map
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 2048,
        dtype = torch.float16,
        load_in_4bit = True,
        device_map = "auto",  # Allow automatic device mapping
    )
    print("Model loaded with auto device mapping")
except Exception as e:
    print(f"Auto device mapping failed with error: {str(e)}")
    print("Trying alternate loading method...")

    # Second attempt: more aggressive memory optimization
    try:
        from accelerate import init_empty_weights
        from transformers import AutoConfig

        # Clear memory again
        torch.cuda.empty_cache()
        gc.collect()

        # Load with maximum memory optimization
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_path,
            max_seq_length = 1024,  # Reduced sequence length to save memory
            dtype = torch.float16,
            load_in_4bit = True,
            device_map = "balanced",  # Balanced strategy instead of auto
            offload_folder = "offload",  # Enable disk offloading if needed
            llm_int8_enable_fp32_cpu_offload = True,  # Enable the setting mentioned in the error
        )
        print("Model loaded with balanced device mapping and offloading")
    except Exception as e:
        print(f"Balanced mapping also failed with error: {str(e)}")
        print("Trying with minimum GPU requirements...")

        # Third attempt: minimal GPU usage
        torch.cuda.empty_cache()
        gc.collect()

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_path,
            max_seq_length = 512,  # Even smaller sequence length
            dtype = torch.float16,
            load_in_4bit = True,
            device_map = {"": "cpu"},  # Start with CPU loading
            offload_folder = "offload",
        )

        # Move only essential layers to GPU
        for name, module in model.named_modules():
            if any(layer_name in name for layer_name in ["lm_head", "embed_tokens"]):
                if hasattr(module, "to"):
                    module.to("cuda:0")
        print("Model loaded with minimal GPU usage")

# Print device map to verify where model components are placed
if hasattr(model, "hf_device_map"):
    print("Model device map:")
    for key, value in model.hf_device_map.items():
        print(f"  {key}: {value}")

# Enable inference mode (if we made it this far)
try:
    FastLanguageModel.for_inference(model)
    print("Inference mode enabled")
except Exception as e:
    print(f"Warning: Couldn't enable fast inference mode: {str(e)}")
    print("Continuing with standard inference")

# Define the alpaca prompt format
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# Load the test Excel file
test_excel_path = "/content/Test-Questions.xlsx"  # Your test file
test_df = pd.read_excel(test_excel_path)
print(f"Loaded {len(test_df)} questions from {test_excel_path}")

# Function to extract the model's answer letter
def extract_answer(text):
    patterns = [
        r"(?i)correct answer is ([A-D])",
        r"(?i)answer is ([A-D])",
        r"(?i)option ([A-D])",
        r"(?i)([A-D])\s+is correct",
        r"(?i)([A-D])\.",  # Fallback pattern
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).upper()

    return None

# Setup for results
results = []
correct_count = 0
total_count = len(test_df)
start_time = time.time()

# Determine batch size based on available GPU memory
# Start with small batch size to be safe
batch_size = 1

print("Starting evaluation of all questions...")
# Process questions in smaller batches to save memory
for i, question in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Format the question
        formatted_question = f"{question['Question']}\n\nOptions:\nA. {question['Option A']}\nB. {question['Option B']}\nC. {question['Option C']}\nD. {question['Option D']}\n\nSelect the correct answer and explain why."

        # Tokenize the input
        inputs = tokenizer(
            [
                alpaca_prompt.format(
                    formatted_question,
                    "",
                )
            ],
            return_tensors = "pt"
        )

        # Move inputs to the appropriate device (could be GPU or CPU depending on model loading)
        if torch.cuda.is_available() and next(model.parameters()).is_cuda:
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # Generate with reduced parameters to save memory
        with torch.no_grad():  # Ensure no gradients are tracked to save memory
            outputs = model.generate(
                **inputs,
                max_new_tokens = 128,  # Reduced from 256
                temperature = 0.3,
                use_cache = True,
                do_sample = False,  # Deterministic generation to save memory
            )

        if torch.cuda.is_available():
            # Move output back to CPU to free GPU memory
            outputs = outputs.cpu()

        result = tokenizer.batch_decode(outputs)[0]

        # Free memory
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Extract the response part
        response = result.split("### Response:")[-1].strip()

        # Extract the predicted answer
        predicted_option = extract_answer(response)

        # Check if correct
        is_correct = predicted_option == question["Correct Option"] if predicted_option else False
        if is_correct:
            correct_count += 1

        # Store result
        results.append({
            "question_number": i + 1,
            "question": question["Question"],
            "correct_option": question["Correct Option"],
            "predicted_option": predicted_option if predicted_option else "Undetermined",
            "is_correct": is_correct,
            "response": response[:300] + "..." if len(response) > 300 else response
        })

    except Exception as e:
        print(f"\nError processing question {i+1}: {str(e)}")
        # Store error result
        results.append({
            "question_number": i + 1,
            "question": question["Question"],
            "correct_option": question["Correct Option"],
            "predicted_option": "ERROR",
            "is_correct": False,
            "response": f"Error: {str(e)}"
        })

# Calculate elapsed time
elapsed_time = time.time() - start_time
avg_time_per_question = elapsed_time / total_count

# Calculate accuracy (only on non-error results)
valid_results = [r for r in results if r["predicted_option"] != "ERROR"]
correct_count = sum(1 for r in valid_results if r["is_correct"])
valid_count = len(valid_results)
accuracy = (correct_count / valid_count) * 100 if valid_count > 0 else 0

# Create results DataFrame
results_df = pd.DataFrame(results)

# Save results to Excel file
output_file = "/content/model_evaluation_results.xlsx"
try:
    results_df.to_excel(output_file, index=False)
    excel_saved = True
except Exception as e:
    print(f"Warning: Could not save to Excel: {str(e)}")
    # Try saving to CSV instead
    csv_file = "/content/model_evaluation_results.csv"
    results_df.to_csv(csv_file, index=False)
    output_file = csv_file
    excel_saved = False

# Print summary
print("\n" + "="*50)
print("EVALUATION SUMMARY")
print("="*50)
print(f"Total questions evaluated: {total_count}")
print(f"Valid predictions: {valid_count}")
print(f"Correct answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Total evaluation time: {elapsed_time:.2f} seconds")
print(f"Average time per question: {avg_time_per_question:.2f} seconds")
print(f"Detailed results saved to: {output_file}")
print("="*50)

# Print a few example predictions
print("\nEXAMPLE PREDICTIONS:")
examples_to_show = min(5, len(results))
for i in range(examples_to_show):
    result = results[i]
    print(f"\nQuestion {result['question_number']}: {result['question']}")
    print(f"Correct Answer: {result['correct_option']}")
    print(f"Predicted Answer: {result['predicted_option']}")
    print(f"Result: {'✓ CORRECT' if result['is_correct'] else '✗ INCORRECT'}")

# Add summary stats to the file for easy reference
if excel_saved:
    summary_data = {
        "Metric": ["Total Questions", "Valid Predictions", "Correct Answers", "Accuracy", "Evaluation Time", "Avg Time per Question"],
        "Value": [
            total_count,
            valid_count,
            correct_count,
            f"{accuracy:.2f}%",
            f"{elapsed_time:.2f} seconds",
            f"{avg_time_per_question:.2f} seconds"
        ]
    }
    summary_df = pd.DataFrame(summary_data)

    try:
        # Write summary to a separate sheet in the same Excel file
        with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
    except Exception as e:
        print(f"Note: Could not add summary sheet to Excel: {str(e)}")
        print("Summary data is displayed in console output only")

print(f"\nEvaluation complete. Results saved to {output_file}")

CUDA available: True
Available GPU memory: 14.74 GB
Current GPU memory usage: 7.41 GB
Loading model from saved checkpoint...
==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load /content/outputs/checkpoint-120 as a legacy tokenizer.


Auto device mapping failed with error: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 100016 has 14.73 GiB memory in use. Of the allocated memory 14.51 GiB is allocated by PyTorch, and 64.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Trying alternate loading method...
==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled 

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

By using https://translate.google.com/ we get
```
Korean music is classified into many types of music genres.

This genre is classified into different music genres such as pop songs,

rock songs, classical songs and pop songs, music groups consisting of drums, fans, instruments and singers
```

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [20]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
    [
        alpaca_prompt.format(
            # "Describe the planet Earth extensively.", # instruction
            "지구를 광범위하게 설명하세요.",
            "",  # output - leave this blank for generation!
        ),
    ],
    return_tensors="pt",
).to("cuda")


from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs, streamer=text_streamer, max_new_tokens=128, repetition_penalty=0.1
)

By using https://translate.google.com/ we get
```
Earth refers to all things including natural disasters such as local derailment

and local depletion that occur in one space along with the suppression of water, gases, and living things.

Most of the Earth's water comes from oceans, atmospheric water, underground water layers, and rivers and rivers.
```

Yikes the language model is a bit whacky! Change the temperature and using sampling will definitely make the output much better!

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit=load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q5_k_m", token = "")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
